# Getting Data

## What is FastF1 ? :

FastF1 API gives you access to F1 lap timing, car telemetry and position, tyre data, weather data, the event schedule and session results.



## Libraries

In [ ]:
import fastf1
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

## Loading Session | Event

In [ ]:
session = fastf1.get_session(2026, 1, 'R') # Can load by name or location also but beware
session.name # Get "Race"
session.event # Accessible like pandas

In [ ]:
event = fastf1.get_event(2026,1)
event

### Lighter Loading :
Allows loading only the data we want :





In [ ]:
session.load(laps=False, telemetry=False, weather=False, messages=False)

An event is a race weekend or a testing event, and consists of multiple sessions
- Session : Race, Qualifying, Free Practice
- Event : Whole weekend
- EventSchedule : Whole Year

In [ ]:
schedule = fastf1.get_event_schedule(2026)
gp_2 = schedule.get_event_by_round(2)
gp_2

### Results of a Race :

In [ ]:
session.load()
session.results

### Session results columns

In [ ]:
session.results.columns

**Features** :
-  `DriverNumber` : Their number displayed on their car
-  `BroadcastName` : Their name displayed on TV
-  `Abbreviation` : The drivers' names, abbreviated
-  `DriverId` : Driver surname
-  `TeamName` : Driver's full team name
-  `TeamColor` : Team color for display
-  `FirstName` : First name
-  `LastName` : Last name
-  `HeadshotUrl` : Link
-  `CountryCode` : Identifier code for a country
-  `Position` : Finishing position
-  `ClassifiedPosition` : Official finishing position
-  `GridPosition` : Starting race position
-  `Q1` : Qualifying 1 : (6 in 2026) 5 of the slowest drivers eliminated
-  `Q2` : Qualifying 2 : (6 in 2026) 5 of the slowest drivers eliminated
-  `Q3` : Qualifying 3 : 10 fastest drivers, determines the starting race position
-  `Time` : Race time taken
-  `Status` : Race end status (Finished, Lapped, Did not finish...)
-  `Points` : Points earned for the championship
-  `Laps` : Laps completed

Example for the top 10 drivers through Q3 (works like pandas)

In [ ]:
session.results.iloc[0:10].loc[:, ['Abbreviation', 'Q3']]

### Laps 

In [ ]:
session.laps


**Laps Features**
- `Time` : Time for the winner and time comparison with the others
- `Driver` : Driver abbreviation
- `DriverNumber` : Driver number displayed on TV
- `LapTime` : Time of a lap
- `LapNumber` : Number of the lap
- `Stint` : Stint number
- `PitInTime` : Time when the driver pits in
- `PitOutTime` : Time when the driver pits out during the lap
- `Sector1-2-3Time` : Duration of the sector 
- `Sector1-2-3Time` : Moment when the sector finishes
- `SpeedI1-I2` : Speed in km/h at the intermediate point (center of sector 1-2)
- `SpeedFL` : Speed in km/h at the finish line
- `SpeedST` : Speed at the speed trap, on the longest straight
- `IsPersonalBest` : Whether the lap is the driver's personal best
- `Compound` : Type of tyre during the lap : SOFT, MEDIUM, HARD, INTERMEDIATE, WET, UNKNOWN/NAN
- `TyreLife` : Age of the tyre in number of laps
- `FreshTyre` : Whether the tyre was fresh or not
- `Team` : Team of the driver
- `LapStartTime` : Time during the session when the lap starts
- `LapStartDate` : Day when the lap starts
- `TrackStatus` : Track status during lap : 1 Green Flag, 2 Yellow Flag, 4 Safety Car, 5 Red Flag, 6 Virtual Safety Car, 7 End of Virtual Safety Car
- `Position` : Position of the driver during the lap
- `Deleted` : Whether the lap was deleted by the stewards
- `DeletedReason` : Reason for deletion
- `FastF1Generated` : Data generated by FastF1 and not official
- `IsAccurate` : Checks if Time and LapStartTime are correlated

### Specific Lap selected

In [ ]:
fastest_lap = session.laps.pick_fastest()
fastest_lap[['Time','Driver']]

## Qualifying :

In [ ]:
qualifying = fastf1.get_session(2026, 1, 'Q')
qualifying.load()



In [ ]:
qualifying.results

The qualifying session gives us all the qualifying times and the qualified drivers.

## Weather 

In [ ]:
session.weather_data

In [ ]:
session.weather_data.info()

Contains the weather information during the session (here the race)
- `Time` : Moment when the weather was measured
- `AirTemp` : Air temperature during the period
- `Humidity` : Percentage of humidity
- `Pressure` : Pressure in mbar
- `Rainfall` : Whether it is raining or not
- `TrackTemp` : Temperature of the track
- `WindDirection` : In degrees : 90 North, 180 East, 270 South, 360 West
- `WindSpeed` : Wind speed in m/s


## How the cache works

All HTTP requests performed by FastF1 go through its caching and rate-limiting system.  
By default, the cache is enabled to speed up scripts and to avoid hitting the API servers' rate limits.

**Preferred ways to set the cache** :

1. A call to enable_cache()
2. The FASTF1_CACHE environment variable
3. An OS-dependent cache directory

Two cache levels:
- **Stage 1 – Raw requests**:  every downloaded response (HTTP GET) is stored locally. It expires after a while so the data gets refreshed.
- **Stage 2 – Parsed data**:  already-built DataFrames are saved (`.ff1pkl`), which skips the expensive parsing step. Only used by some functions, mainly `session.load()`.

→ The first load is slow (download + parsing); the next ones are almost instant.



In [ ]:
# Must be called after imports
fastf1.Cache.enable_cache('path/to/cache')

# %LOCALAPPDATA%\Temp\fastf1 -> default folder